# 03 · Evaluation — Amazon Movies & TV (PySpark ALS)

Compares three models using held-out test ratings:
- **Global Mean** baseline  
- **Popularity** baseline  
- **ALS** model  

Metrics: **RMSE / MAE** · **Precision@K / Recall@K** · **Effect of rank**

In [ ]:
# ── Update to your Databricks volume path ─────────────────────────────────────
DATA_DIR = "/Volumes/workspace/ds_amazonrec/data"
# ─────────────────────────────────────────────────────────────────────────────

## 1 · Load, Filter, and Index

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, FloatType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import time

# Reviews split into parts for upload — Spark merges them automatically
raw_reviews = spark.read.json(f"{DATA_DIR}/reviews_part*.jsonl")

# Explicit schema avoids duplicate-column errors in metadata JSON
meta_schema = StructType([
    StructField("parent_asin", StringType(), True),
    StructField("title",       StringType(), True),
])
raw_meta = spark.read.schema(meta_schema).json(f"{DATA_DIR}/meta_Movies_and_TV.jsonl")

reviews = raw_reviews.select(
    F.col("user_id"),
    F.col("parent_asin").alias("product_id"),
    F.col("rating").cast("float"),
)

products = raw_meta.select(
    F.col("parent_asin").alias("product_id"),
    F.col("title").alias("name"),
).dropDuplicates(["product_id"])

print(f"Raw: {reviews.count():,} reviews")

In [ ]:
# 5-core filter — keep users and products with >= 5 interactions
def apply_k_core(df, k=5, max_iter=4):
    for i in range(max_iter):
        product_counts = df.groupBy("product_id").count()
        df = df.join(product_counts.filter(F.col("count") >= k).select("product_id"), "product_id")
        user_counts = df.groupBy("user_id").count()
        df = df.join(user_counts.filter(F.col("count") >= k).select("user_id"), "user_id")
        print(f"Iter {i+1} done")
    return df

reviews_filtered = apply_k_core(reviews)
print(f"After 5-core: {reviews_filtered.count():,} reviews")

In [ ]:
# dense_rank replaces StringIndexer — avoids model size limit on Serverless
user_mapping = reviews_filtered.select("user_id").distinct() \
    .withColumn("userIdx", (F.dense_rank().over(Window.orderBy("user_id")) - 1).cast("integer"))

product_mapping = reviews_filtered.select("product_id").distinct() \
    .withColumn("productIdx", (F.dense_rank().over(Window.orderBy("product_id")) - 1).cast("integer"))

# Join indices back to reviews
indexed = reviews_filtered \
    .join(user_mapping, "user_id") \
    .join(product_mapping, "product_id")

# Ensure integer type for ALS
indexed = indexed.withColumn("userIdx",    F.col("userIdx").cast("integer")) \
                 .withColumn("productIdx", F.col("productIdx").cast("integer"))

# Reverse mappings: integer → original string ID
idx_to_product = {r["productIdx"]: r["product_id"] for r in product_mapping.collect()}

train, test = indexed.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"Train: {train.count():,} | Test: {test.count():,}")

## 2 · Baseline Models

In [ ]:
evaluator     = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="mae")

global_mean = train.select(F.mean("rating")).first()[0]

# Global Mean baseline — predict the same value for every user/product
test_mean = test.withColumn("prediction", F.lit(float(global_mean)))
rmse_mean = evaluator.evaluate(test_mean)
mae_mean  = evaluator_mae.evaluate(test_mean)
print(f"Global Mean  — RMSE: {rmse_mean:.4f}  MAE: {mae_mean:.4f}")

# Popularity baseline — predict each product's average rating
product_avg = train.groupBy("productIdx").agg(F.mean("rating").alias("prediction"))
test_pop    = test.join(product_avg, "productIdx", "left").fillna(global_mean, subset=["prediction"])
rmse_pop    = evaluator.evaluate(test_pop)
mae_pop     = evaluator_mae.evaluate(test_pop)
print(f"Popularity   — RMSE: {rmse_pop:.4f}  MAE: {mae_pop:.4f}")

## 3 · Train ALS Model

In [ ]:
RANK = 50

als = ALS(
    rank=RANK, maxIter=10, regParam=0.1,
    userCol="userIdx", itemCol="productIdx", ratingCol="rating",
    implicitPrefs=False, coldStartStrategy="drop", seed=42
)

t0 = time.time()
als_model = als.fit(train)
print(f"ALS trained in {time.time()-t0:.1f}s  (rank={RANK})")

predictions = als_model.transform(test)
rmse_als    = evaluator.evaluate(predictions)
mae_als     = evaluator_mae.evaluate(predictions)
print(f"ALS (rank={RANK}) — RMSE: {rmse_als:.4f}  MAE: {mae_als:.4f}")

## 4 · Comparison Table

In [ ]:
results = pd.DataFrame([
    {"Model": "Global Mean",         "RMSE": rmse_mean, "MAE": mae_mean},
    {"Model": "Popularity",          "RMSE": rmse_pop,  "MAE": mae_pop},
    {"Model": f"ALS (rank={RANK})",  "RMSE": rmse_als,  "MAE": mae_als},
]).set_index("Model").round(4)
results["RMSE Δ vs Mean"] = (results["RMSE"] - rmse_mean).round(4)
print(results.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ["#aec6cf", "#ffb347", "#77dd77"]
for ax, metric in zip(axes, ["RMSE", "MAE"]):
    bars = ax.bar(results.index, results[metric], color=colors, edgecolor="white", width=0.5)
    ax.set_title(metric, fontsize=13)
    ax.set_ylim(0, results[metric].max() * 1.25)
    for bar, val in zip(bars, results[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=10)
plt.suptitle("Model Comparison — Amazon Movies & TV", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5 · Top-N Recommendation Quality (Precision@K / Recall@K)

Unity Catalog does not support `recommendForAllUsers()`. We use crossJoin + Window instead.

In [ ]:
K_REC = 10
SAMPLE_FRACTION = 0.01  # 1% of users on Databricks; use 0.1 locally on small data

sample_users = test.select("userIdx").distinct().sample(SAMPLE_FRACTION, seed=42)
all_items    = train.select("productIdx").distinct()
candidates   = sample_users.crossJoin(all_items)
window       = Window.partitionBy("userIdx").orderBy(F.col("prediction").desc())

user_recs = (
    als_model.transform(candidates)
    .filter(F.col("prediction").isNotNull())
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") <= K_REC)
    .select("userIdx", "productIdx")
)

test_actuals = test.join(sample_users, "userIdx").select("userIdx", "productIdx")

# Hits = recommended items that appear in test actuals
hits = user_recs.join(test_actuals, ["userIdx", "productIdx"]).groupBy("userIdx").agg(
    F.count("*").alias("n_hits")
)
rec_counts = user_recs.groupBy("userIdx").agg(F.count("*").alias("n_rec"))
act_counts = test_actuals.groupBy("userIdx").agg(F.count("*").alias("n_actual"))

(
    rec_counts
    .join(act_counts, "userIdx")
    .join(hits, "userIdx", "left")
    .fillna(0, subset=["n_hits"])
    .agg(
        F.round(F.mean(F.col("n_hits") / F.col("n_rec")),    4).alias(f"Precision@{K_REC}"),
        F.round(F.mean(F.col("n_hits") / F.col("n_actual")), 4).alias(f"Recall@{K_REC}"),
    )
).show()

## 6 · Effect of ALS Rank on RMSE

In [ ]:
rank_values  = [5, 10, 20, 30, 50, 75, 100]
rmse_by_rank = []
time_by_rank = []

for rank in rank_values:
    t0 = time.time()
    m  = ALS(
        rank=rank, maxIter=10, regParam=0.1,
        userCol="userIdx", itemCol="productIdx", ratingCol="rating",
        implicitPrefs=False, coldStartStrategy="drop", seed=42
    ).fit(train)
    elapsed = time.time() - t0
    r = evaluator.evaluate(m.transform(test))
    rmse_by_rank.append(r)
    time_by_rank.append(elapsed)
    print(f"rank={rank:3d}  RMSE={r:.4f}  time={elapsed:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rank_values, rmse_by_rank, marker="o", color="tomato", label="ALS")
axes[0].axhline(rmse_mean, linestyle="--", color="gray",    label="Global Mean")
axes[0].axhline(rmse_pop,  linestyle="--", color="#ffb347", label="Popularity")
axes[0].set_xlabel("rank"); axes[0].set_ylabel("RMSE"); axes[0].set_title("RMSE vs rank")
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(rank_values, time_by_rank, marker="s", color="steelblue")
axes[1].set_xlabel("rank"); axes[1].set_ylabel("Train time (s)"); axes[1].set_title("Train time vs rank")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7 · Summary

In [ ]:
best_rank = rank_values[rmse_by_rank.index(min(rmse_by_rank))]
best_rmse = min(rmse_by_rank)

print("=" * 58)
print("SUMMARY — Amazon Movies & TV, PySpark ALS")
print("=" * 58)
print(f"Global Mean RMSE          : {rmse_mean:.4f}")
print(f"Popularity RMSE           : {rmse_pop:.4f}")
print(f"ALS best RMSE             : {best_rmse:.4f}  (rank={best_rank})")
print(f"Improvement over Mean     : {(rmse_mean - best_rmse)/rmse_mean*100:.1f}%")
print("=" * 58)
print()
print("Possible next steps:")
print("  - Try implicitPrefs=True with binarised ratings (>=4 → 1, else 0)")
print("  - Add content features from product metadata")
print("  - Tune regParam with cross-validation")